# Kaleido — Pipeline 1 v5.2: Essay Generation (Direction Funnel)

**What changed from v5:** The single `generate_essay()` call that dumped all 126 directions into one massive prompt is replaced with a 4-stage funnel:

1. **Reasoner** (o3-mini, `reasoning_effort="high"`) — Analyzes the question. Identifies 2-3 activated poles. Sees only Reasoner.md (7 poles). Sees NO directions — pure conceptual thinking.
2. **Lookup** (Python, instant) — Maps activated poles → candidate directions. Deterministic filter: 126 → ~18-36.
3. **Matcher** (o3-mini, `reasoning_effort="high"`) — Receives Matcher.md as standing instructions + ~20 filtered candidates WITH full ARGUMENT + LOGIC. Picks 2-3 best-fit directions.
4. **Generator** (o3-mini, `reasoning_effort="high"`) — Receives Writer.md + selected directions + Reference Sec 2 + Sec 3. Writes essay with ONLY the selected directions in context. Same output format as v5.

**Why:** The Reasoner reasons without distraction. The Lookup narrows mechanically. The Matcher compares deeply (full LOGIC, not just arguments). The Generator writes focused prose. No single stage is overloaded.

### Before running the pipeline, paste this to SQL Edit in Supabase UI to create tables for data generated to be populated:

create extension if not exists vector;

  

create table essays (

essay_id uuid primary key,

question text,

structure_type text,

word_count int,

batch_id uuid,

created_at timestamp default now(),

updated_at timestamp default now()

);

  

create table sentences (

sentence_id uuid primary key,

canonical_text text,

embedding vector(384),

rhetoric_tag text,

direction_tag text,

word_count int,

batch_id uuid,

created_at timestamp default now(),

updated_at timestamp default now()

);

  

create table essay_sentences (

id uuid primary key,

essay_id uuid references essays(essay_id),

sentence_id uuid references sentences(sentence_id),

paragraph_type text,

"order" int,

created_at timestamp default now(),

updated_at timestamp default now()

);

  

create table lexical_items (

lexical_id uuid primary key,

sentence_id uuid references sentences(sentence_id),

phrase text,

embedding vector(384),

created_at timestamp default now(),

updated_at timestamp default now()

);

  

create table syntax_items (

syntax_id uuid primary key,

pattern text,

embedding vector(384),

created_at timestamp default now(),

updated_at timestamp default now()

);

  

create table sentence_syntax (

id uuid primary key,

sentence_id uuid references sentences(sentence_id),

syntax_id uuid references syntax_items(syntax_id),

created_at timestamp default now(),

updated_at timestamp default now()

);

## 🔧 Station 0 — Install Dependencies

In [ ]:
!pip install -q openai supabase sentence-transformers pydantic google-colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.2 MB/s eta 0:00:00


## 🔑 Station 0b — Load Secrets & Connect

In [ ]:
import openai
from supabase import create_client

# Paste your credentials directly here
OPENAI_API_KEY = ""
SUPABASE_URL   = "https://yxxmjptlkeewcabczlgt.supabase.co"
SUPABASE_KEY   = "sb_publishable_VcS5cSLESM977M6IQpcscA_TsqUx5Es"

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
supabase      = create_client(SUPABASE_URL, SUPABASE_KEY)

print('✅ OpenAI client ready')
print('✅ Supabase client ready')

✅ OpenAI client ready
✅ Supabase client ready


## 📐 Station 0c — Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading embedding model (all-MiniLM-L6-v2, 384-dim)...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model ready')

def embed(text: str) -> list:
    """Embed a single string. Returns list of 384 floats."""
    return embed_model.encode(text).tolist()

Loading embedding model (all-MiniLM-L6-v2, 384-dim)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready


## 📋 Station 0d — Valid Tags (Guard Rails)

In [ ]:
# All 126 valid direction IDs from Reference-v2.md Section 1
VALID_DIRECTION_TAGS = {
    # Pair 1: Material <-> Sustainable (8)
    'material_sustainable_causal_neg', 'sustainable_material_causal_neg',
    'material_sustainable_tradeoff', 'material_sustainable_synergy',
    'material_sustainable_instrument', 'material_sustainable_blocking',
    'material_sustainable_transformation', 'material_sustainable_feedback_neg',
    # Pair 2: Material <-> Individual (6)
    'individual_material_causal_pos', 'material_individual_causal_neg',
    'material_individual_synergy', 'material_individual_instrument',
    'material_individual_tradeoff', 'material_individual_feedback_pos',
    # Pair 3: Material <-> Collective (6)
    'material_collective_instrument', 'collective_material_instrument',
    'material_collective_tradeoff', 'collective_material_blocking',
    'material_collective_synergy', 'material_collective_feedback_neg',
    # Pair 4: Material <-> Progress (6)
    'material_progress_instrument', 'progress_material_causal_pos',
    'material_progress_feedback_pos', 'material_progress_tradeoff',
    'progress_material_spillover_neg', 'material_progress_transformation',
    # Pair 5: Material <-> Preservation (6)
    'material_preservation_causal_neg', 'material_preservation_instrument_pos',
    'material_preservation_tradeoff', 'material_preservation_paradox',
    'preservation_material_instrument', 'material_preservation_spillover_neg',
    # Pair 6: Material <-> Flourishing (7)
    'material_flourishing_causal_neg', 'material_flourishing_causal_pos',
    'material_flourishing_tradeoff', 'material_flourishing_instrument',
    'flourishing_material_instrument', 'material_flourishing_blocking',
    'material_flourishing_feedback_neg',
    # Pair 7: Sustainable <-> Individual (6)
    'sustainable_individual_causal_pos', 'individual_sustainable_causal_neg',
    'individual_sustainable_insufficient', 'sustainable_individual_instrument',
    'individual_sustainable_tradeoff', 'sustainable_individual_spillover',
    # Pair 8: Sustainable <-> Collective (6)
    'sustainable_collective_instrument', 'collective_sustainable_instrument',
    'sustainable_collective_blocking', 'collective_sustainable_causal_pos',
    'sustainable_collective_tradeoff', 'collective_sustainable_feedback_pos',
    # Pair 9: Sustainable <-> Progress (6)
    'progress_sustainable_causal_pos', 'progress_sustainable_causal_neg',
    'sustainable_progress_instrument', 'progress_sustainable_spillover_neg',
    'sustainable_progress_tradeoff', 'progress_sustainable_transformation',
    # Pair 10: Sustainable <-> Preservation (5)
    'sustainable_preservation_synergy', 'preservation_sustainable_instrument',
    'sustainable_preservation_causal_pos', 'preservation_sustainable_causal_neg',
    'sustainable_preservation_tradeoff',
    # Pair 11: Sustainable <-> Flourishing (5)
    'sustainable_flourishing_causal_pos', 'flourishing_sustainable_instrument',
    'sustainable_flourishing_synergy', 'sustainable_flourishing_tradeoff',
    'flourishing_sustainable_causal_neg',
    # Pair 12: Individual <-> Collective (7)
    'individual_collective_tradeoff', 'collective_individual_instrument',
    'individual_collective_blocking', 'collective_individual_blocking',
    'individual_collective_synergy', 'collective_individual_causal_pos',
    'individual_collective_feedback_neg',
    # Pair 13: Individual <-> Progress (6)
    'progress_individual_causal_pos', 'individual_progress_causal_pos',
    'progress_individual_paradox', 'progress_individual_spillover_neg',
    'progress_individual_instrument', 'individual_progress_instrument',
    # Pair 14: Individual <-> Preservation (5)
    'individual_preservation_causal_pos', 'preservation_individual_causal_pos',
    'individual_preservation_insufficient', 'individual_preservation_tradeoff',
    'preservation_individual_instrument',
    # Pair 15: Individual <-> Flourishing (5)
    'individual_flourishing_synergy', 'individual_flourishing_tradeoff',
    'flourishing_individual_instrument', 'individual_flourishing_causal_neg',
    'flourishing_individual_blocking',
    # Pair 16: Collective <-> Progress (6)
    'collective_progress_instrument', 'progress_collective_instrument',
    'collective_progress_tradeoff', 'progress_collective_spillover_neg',
    'collective_progress_synergy', 'progress_collective_blocking',
    # Pair 17: Collective <-> Preservation (5)
    'collective_preservation_instrument', 'preservation_collective_instrument',
    'collective_preservation_tradeoff', 'collective_preservation_spillover_neg',
    'preservation_collective_causal_pos',
    # Pair 18: Collective <-> Flourishing (5)
    'collective_flourishing_instrument', 'flourishing_collective_instrument',
    'collective_flourishing_tradeoff', 'collective_flourishing_causal_pos',
    'flourishing_collective_blocking',
    # Pair 19: Progress <-> Preservation (6)
    'progress_preservation_causal_neg', 'progress_preservation_instrument_pos',
    'preservation_progress_blocking', 'preservation_progress_informing',
    'progress_preservation_tradeoff', 'progress_preservation_transformation',
    # Pair 20: Progress <-> Flourishing (8)
    'progress_flourishing_causal_pos', 'progress_flourishing_causal_neg',
    'progress_flourishing_instrument', 'progress_flourishing_spillover_neg',
    'progress_flourishing_blocking', 'progress_flourishing_transformation',
    'flourishing_progress_instrument', 'progress_flourishing_feedback_neg',
    # Pair 21: Preservation <-> Flourishing (6)
    'preservation_flourishing_causal_pos', 'flourishing_preservation_instrument',
    'preservation_flourishing_tradeoff', 'flourishing_preservation_causal_pos',
    'preservation_flourishing_synergy', 'preservation_flourishing_blocking',
}

# All 33 valid rhetoric tags from Reference-v2.md Section 2
VALID_RHETORIC_TAGS = {
    'paraphrase_question', 'outline_statement', 'clear_thesis_statement',
    'topic_sentence', 'explanation', 'example_1', 'example_2',
    'mini_conclusion', 'link_to_thesis', 'opinion_topic_sentence',
    'reason_1', 'reason_2', 'synthesis',
    'acknowledge_opposing_view', 'concession', 'rebuttal', 'return_to_thesis',
    'problem_1', 'problem_2', 'solution_1', 'solution_2',
    'cause_1', 'cause_2', 'effect_1', 'effect_2',
    'advantage_1', 'advantage_2', 'disadvantage_1', 'disadvantage_2',
    'opinion_statement', 'restate_thesis', 'summary_statement', 'which_outweighs_opinion',
}

VALID_STRUCTURE_TYPES = {
    'structure_1', 'structure_2', 'structure_3', 'structure_4',
    'structure_5', 'structure_6', 'structure_7'
}

VALID_PARAGRAPH_TYPES = {
    'introduction', 'body_1', 'body_2', 'body_3', 'conclusion'
}

print('✅ Valid tag sets loaded')
print(f'   Direction tags: {len(VALID_DIRECTION_TAGS)}')
print(f'   Rhetoric tags:  {len(VALID_RHETORIC_TAGS)}')

## 📊 Station 0d-ii — Pole Index & Direction Parser

**New in v5.2:** Builds a lookup table mapping each pole to its directions, and parses Reference.md into a structured dictionary so we can filter directions programmatically before the LLM ever sees them.

In [ ]:
import re

# ── A. Pole Index: maps each pole name → set of direction IDs involving it ──
POLES = ['material', 'sustainable', 'individual', 'collective',
         'progress', 'preservation', 'flourishing']

def build_pole_index(valid_tags):
    """For each pole, collect every direction ID that contains that pole."""
    index = {p: set() for p in POLES}
    for tag in valid_tags:
        for pole in POLES:
            if pole in tag.split('_'):
                index[pole].add(tag)
    return index

POLE_INDEX = build_pole_index(VALID_DIRECTION_TAGS)

print('✅ Pole index built')
for p in POLES:
    print(f'   {p:14s}: {len(POLE_INDEX[p]):3d} directions')


# ── B. Direction Parser: extracts NAME/ARGUMENT/LOGIC from Reference.md ──
def parse_directions_from_reference(reference_text):
    """Parse all 126 directions into {name: {argument, logic, pair}}."""
    directions = {}
    current_pair = ''
    lines = reference_text.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('## PAIR'):
            current_pair = line
        if line.startswith('NAME:'):
            name = line.replace('NAME:', '').strip()
            argument, logic = '', ''
            i += 1
            # Find ARGUMENT
            while i < len(lines) and not lines[i].strip().startswith('ARGUMENT:'):
                i += 1
            if i < len(lines):
                arg_parts = [lines[i].replace('ARGUMENT:', '').strip()]
                i += 1
                while i < len(lines) and not lines[i].strip().startswith('LOGIC:') \
                      and not lines[i].strip().startswith('NAME:') \
                      and lines[i].strip() != '```':
                    if lines[i].strip():
                        arg_parts.append(lines[i].strip())
                    i += 1
                argument = ' '.join(arg_parts)
            # Find LOGIC
            if i < len(lines) and lines[i].strip().startswith('LOGIC:'):
                logic_parts = [lines[i].replace('LOGIC:', '').strip()]
                i += 1
                while i < len(lines) and not lines[i].strip().startswith('NAME:') \
                      and lines[i].strip() != '```' and not lines[i].strip().startswith('##'):
                    if lines[i].strip():
                        logic_parts.append(lines[i].strip())
                    i += 1
                logic = ' '.join(logic_parts)
            directions[name] = {'argument': argument, 'logic': logic, 'pair': current_pair}
            continue
        i += 1
    return directions


# ── C. Reference Splitter: separates Sections 1, 2, 3 ──
def split_reference_sections(reference_text):
    """Split Reference.md into directions (Sec 1), rhetoric tags (Sec 2), templates (Sec 3)."""
    sec2_match = re.search(r'^#\s+SECTION 2', reference_text, re.MULTILINE)
    sec3_match = re.search(r'^#\s+SECTION 3', reference_text, re.MULTILINE)
    sec2_text = reference_text[sec2_match.start():sec3_match.start()] if sec2_match and sec3_match else ''
    sec3_text = reference_text[sec3_match.start():] if sec3_match else ''
    return sec2_text, sec3_text

print('\n✅ Direction parser & reference splitter ready')

## 🏗️ Station 0e — Pydantic Output Schema

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class LexicalItem(BaseModel):
    phrase: str = Field(..., description="Exact phrase from the sentence text")

class SentenceOutput(BaseModel):
    paragraph_type: str = Field(..., description="One of: introduction, body_1, body_2, body_3, conclusion")
    order: int          = Field(..., description="Position of this sentence within its paragraph (1-indexed)")
    text: str           = Field(..., description="Full sentence text, verbatim from the essay")
    rhetoric_tag: str   = Field(..., description="One of the 33 valid rhetoric tags")
    direction_tag: Optional[str] = Field(None, description="One of the 126 valid direction IDs, or null")
    lexical_items: list[LexicalItem] = Field(..., description="At least 2 lexical phrases extracted verbatim")
    syntax_pattern: Optional[str]    = Field(None, description="Free-form syntactic pattern observed in this sentence")

    @field_validator('rhetoric_tag')
    @classmethod
    def validate_rhetoric(cls, v):
        if v not in VALID_RHETORIC_TAGS:
            raise ValueError(f"Invalid rhetoric_tag: '{v}'")
        return v

    @field_validator('direction_tag')
    @classmethod
    def validate_direction(cls, v):
        if v is not None and v not in VALID_DIRECTION_TAGS:
            raise ValueError(f"Invalid direction_tag: '{v}'")
        return v

    @field_validator('paragraph_type')
    @classmethod
    def validate_paragraph_type(cls, v):
        if v not in VALID_PARAGRAPH_TYPES:
            raise ValueError(f"Invalid paragraph_type: '{v}'")
        return v

    @field_validator('lexical_items')
    @classmethod
    def validate_min_lexical(cls, v):
        if len(v) < 2:
            raise ValueError("Each sentence must have at least 2 lexical items")
        return v

class EssayOutput(BaseModel):
    structure_type: str              = Field(..., description="One of: structure_1 through structure_7")
    sentences: list[SentenceOutput]  = Field(..., description="All sentences in essay order")

    @field_validator('structure_type')
    @classmethod
    def validate_structure(cls, v):
        if v not in VALID_STRUCTURE_TYPES:
            raise ValueError(f"Invalid structure_type: '{v}'")
        return v

print('✅ Pydantic schemas ready')

✅ Pydantic schemas ready


## 📂 Station 0f — Upload Framework Documents

Upload **4 files** when prompted:
- `Reasoner.md`
- `Matcher.md`
- `Writer.md`
- `Reference-v2.md`

Each stage receives only the documents it needs — no single stage sees the full set.

In [ ]:
from google.colab import files as _doc_files

REQUIRED_DOCS = {
    'Reasoner.md':     None,
    'Matcher.md':      None,
    'Writer.md':       None,
    'Reference-v2.md': None,
}

print('Upload Reasoner.md, Matcher.md, Writer.md, and Reference-v2.md (select all 4 at once):')
uploaded_docs = _doc_files.upload()

for filename, content_bytes in uploaded_docs.items():
    for key in REQUIRED_DOCS:
        if filename.lower() == key.lower():
            REQUIRED_DOCS[key] = content_bytes.decode('utf-8')
            print(f'  \u2705 {key} loaded ({len(REQUIRED_DOCS[key]):,} characters)')

missing = [k for k, v in REQUIRED_DOCS.items() if v is None]
if missing:
    raise ValueError(f'\u274c Missing documents: {missing}. Re-run this cell and upload all 4 files.')

# \u2500\u2500 Parse directions into structured dict \u2500\u2500
DIRECTION_CATALOG = parse_directions_from_reference(REQUIRED_DOCS['Reference-v2.md'])
print(f'\n\u2705 Parsed {len(DIRECTION_CATALOG)} directions from Reference.md')

# \u2500\u2500 Split Reference into reusable sections \u2500\u2500
REF_SEC2_RHETORIC, REF_SEC3_TEMPLATES = split_reference_sections(REQUIRED_DOCS['Reference-v2.md'])
print(f'   Section 2 (rhetoric tags): {len(REF_SEC2_RHETORIC):,} chars')
print(f'   Section 3 (templates):     {len(REF_SEC3_TEMPLATES):,} chars')

print('\n\u2705 Framework documents loaded and parsed')

## 📂 Station 1 — Upload & Parse Questions

In [ ]:
from google.colab import files
import re

print('Upload your 100_questions_mvp.md file:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
content  = uploaded[filename].decode('utf-8')

# Parse numbered list: "1. Question text"
questions = []
for line in content.splitlines():
    line = line.strip()
    match = re.match(r'^(\d+)\.\s+(.+)$', line)
    if match:
        q_num  = int(match.group(1))
        q_text = match.group(2).strip()
        questions.append({'number': q_num, 'text': q_text})

print(f'\n✅ Parsed {len(questions)} questions')
print('\nFirst 3 questions:')
for q in questions[:3]:
    print(f"  {q['number']}. {q['text']}")

Upload your 100_questions_mvp.md file:


Saving 100_questions_mvp.md to 100_questions_mvp.md

✅ Parsed 100 questions

First 3 questions:
  1. Some people think society could benefit more if more people study business than history. To what extent do you agree or disagree?
  2. Some argue that schools should prioritize life skills such as working in teams and solving problems instead of traditional academics. To what extent do you agree or disagree?
  3. Some people believe the purpose of education should be helping the individual to become useful for society, while others believe it should help individuals to achieve their ambitions. Discuss both views and give your opinion.


## 🤖 Station 2 — Three-Stage Direction Finding + Generation

**v5.2 architecture change:** Instead of dumping all 126 directions into one massive prompt, we now use a 3-step funnel to find the best directions BEFORE generating the essay.

```
Question
  → Reasoner (LLM)       — deep analysis, identifies 2-3 activated poles
  → Lookup   (Python)    — poles → ~18-36 candidate directions (deterministic)
  → Matcher  (LLM)       — reads candidates with full LOGIC, picks 2-3 best
  → Generator (LLM)      — writes essay with ONLY the selected directions in context
```

The Reasoner never sees directions (pure thinking). The Matcher sees ~20 directions with full LOGIC (focused comparison). The Generator sees only 2-3 directions (laser-focused writing).

In [ ]:
import json
import time

# ================================================================
# STAGE 1: REASONER (gpt-4o) — Analyze question, identify poles
# ================================================================
# Uses gpt-4o: stronger conceptual reasoning than o3-mini for this task.
# Sees ONLY the 7 pole definitions. Never sees any direction names.
# Simplified output: just poles + why. No connection_types or argument_shape
# — those split the model's attention away from the real job.

REASONER_SYSTEM = f"""You are a question analyst. Your job is to read an IELTS essay question and identify which 2-3 human forces are in tension.

Below are seven forces (called "poles"). Each one represents a real human need. Every IELTS question is built on a tension between two or more of these.

=== THE SEVEN POLES ===
{REQUIRED_DOCS['Reasoner.md']}

=== YOUR TASK ===
Read the question. Feel the tension. Ask yourself:
- What are people REALLY disagreeing about in this question?
- Which two or three of these poles are pulling in different directions?

=== OUTPUT ===
Return JSON only:
{{
  "question_analysis": "2-3 sentences: what is this question really about? What human tension drives the debate?",
  "activated_poles": ["pole1", "pole2"],
  "pole_reasoning": {{
    "pole1": "one sentence: why this force is at play in this question",
    "pole2": "one sentence: why this force is at play in this question"
  }}
}}

VALID POLES: material, sustainable, individual, collective, progress, preservation, flourishing
Pick 2-3 poles. Think carefully."""


def reason_about_question(question_text, max_retries=2):
    """Stage 1: gpt-4o analyzes question -> activated poles. No directions shown."""
    for attempt in range(max_retries + 1):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": REASONER_SYSTEM},
                    {"role": "user",   "content": f"Analyze this IELTS question:\n\n{question_text}"}
                ],
                max_tokens=1000,
                temperature=0.3,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            if not content or not content.strip():
                raise ValueError("Model returned empty response")
            result = json.loads(content)
            # Validate poles
            for p in result.get('activated_poles', []):
                if p not in POLES:
                    raise ValueError(f"Invalid pole: '{p}'")
            if len(result.get('activated_poles', [])) < 2:
                raise ValueError("Must identify at least 2 poles")
            return result
        except Exception as e:
            if attempt < max_retries:
                print(f'    ⚠️ Reasoner attempt {attempt+1}: {e}. Retrying...')
                time.sleep(1)
            else:
                print(f'    ❌ Reasoner failed: {e}')
                return None


# ================================================================
# STAGE 1.5: LOOKUP — Deterministic pole -> direction filtering
# ================================================================
# Pure Python. Takes activated poles, returns only directions
# where BOTH poles in the direction name are activated.

def lookup_directions(activated_poles, direction_catalog, pole_index):
    """Filter directions to only those connecting two activated poles."""
    candidate_tags = set()
    for pole in activated_poles:
        candidate_tags |= pole_index.get(pole, set())

    # Keep only directions where BOTH poles in the name are activated
    filtered = {}
    for tag in candidate_tags:
        parts = tag.split('_')
        matching = [p for p in activated_poles if p in parts]
        if len(matching) >= 2:
            if tag in direction_catalog:
                filtered[tag] = direction_catalog[tag]

    return filtered


# ================================================================
# STAGE 2: MATCHER (o3-mini) — Select 2-3 best from candidates
# ================================================================
# Sees ~18-36 filtered directions WITH full ARGUMENT + LOGIC.
# Also sees the Reasoner's analysis for context.

def build_matcher_prompt(reasoner_result, filtered_directions):
    """Build Matcher system prompt with filtered directions."""
    dir_parts = []
    for name, info in sorted(filtered_directions.items()):
        dir_parts.append(
            f"  {name}\n"
            f"    ARGUMENT: {info['argument']}\n"
            f"    LOGIC: {info['logic']}"
        )
    dir_block = '\n\n'.join(dir_parts)

    # Pull the Reasoner's pole reasoning into a readable block
    pole_notes = []
    for pole, reason in reasoner_result.get('pole_reasoning', {}).items():
        pole_notes.append(f"  {pole}: {reason}")
    pole_block = '\n'.join(pole_notes)

    return f"""{REQUIRED_DOCS['Matcher.md']}

=== ANALYST'S FINDINGS ===
{reasoner_result['question_analysis']}

Activated poles:
{pole_block}

=== CANDIDATE DIRECTIONS ({len(filtered_directions)} candidates) ===
{dir_block}

=== OUTPUT FORMAT ===
Return ONLY valid JSON:
{{
  "question_type": "discuss_both_views | agree_disagree | advantages_disadvantages | problems_solutions | causes_effects_solutions",
  "structure_type": "structure_N",
  "selected_directions": [
    {{
      "direction_id": "exact_name_from_candidates",
      "assigned_to": "body_1 | body_2 | body_3",
      "rationale": "one sentence: why this direction + its LOGIC naturally answers this aspect"
    }}
  ],
  "freestyle_paragraphs": [],
  "essay_plan": "2-3 sentences: what Body 1 argues, what Body 2 argues, how they connect"
}}

CRITICAL: Every direction_id must EXACTLY match one from the candidates above."""


def match_directions(question_text, reasoner_result, filtered_directions, max_retries=2):
    """Stage 2: o3-mini selects 2-3 best directions from filtered candidates."""
    system = build_matcher_prompt(reasoner_result, filtered_directions)

    for attempt in range(max_retries + 1):
        try:
            response = openai_client.chat.completions.create(
                model="o3-mini",
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": (
                        f"Select the best-fit directions for this question:\n\n{question_text}\n\n"
                        "Apply the intuition test to each candidate. Output only JSON."
                    )}
                ],
                reasoning_effort="high",
                max_completion_tokens=8000,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            if not content or not content.strip():
                raise ValueError("Model returned empty response")
            result = json.loads(content)
            # Validate direction IDs against filtered set
            for d in result.get('selected_directions', []):
                did = d['direction_id']
                if did not in filtered_directions:
                    raise ValueError(f"Matcher picked '{did}' -- not in filtered candidates")
                if did not in VALID_DIRECTION_TAGS:
                    raise ValueError(f"Invalid direction: '{did}'")
            if result.get('structure_type') not in VALID_STRUCTURE_TYPES:
                raise ValueError(f"Invalid structure_type: '{result.get('structure_type')}'")
            return result
        except Exception as e:
            if attempt < max_retries:
                print(f'    ⚠️ Matcher attempt {attempt+1}: {e}. Retrying...')
                time.sleep(1)
            else:
                print(f'    ❌ Matcher failed: {e}')
                return None


# ================================================================
# STAGE 3: GENERATOR (gpt-4o) — Write essay with focused context
# ================================================================

def generate_essay(question_text, max_retries=3, verbose=True):
    """
    Full pipeline: Reasoner (gpt-4o) -> Lookup -> Matcher (o3-mini) -> Generator (gpt-4o).
    Output is the same EssayOutput as v5.
    """

    # -- Stage 1: Reasoner (gpt-4o) --
    if verbose:
        print('    [1/4] Reasoning about question (gpt-4o)...', end=' ')
    reasoner_result = reason_about_question(question_text)
    if reasoner_result is None:
        print('FAILED')
        return None
    poles = reasoner_result['activated_poles']
    if verbose:
        print(f'OK -> poles: {poles}')

    # -- Stage 1.5: Lookup (Python) --
    if verbose:
        print('    [2/4] Looking up candidate directions...', end=' ')
    filtered = lookup_directions(poles, DIRECTION_CATALOG, POLE_INDEX)
    if verbose:
        print(f'OK -> {len(filtered)} candidates (from 126)')

    # Safety: if lookup returns too few, widen to single-pole matches
    if len(filtered) < 6:
        if verbose:
            print(f'    ⚠️  Only {len(filtered)} -- widening to single-pole matches')
        for pole in poles:
            for tag in POLE_INDEX.get(pole, set()):
                if tag in DIRECTION_CATALOG:
                    filtered[tag] = DIRECTION_CATALOG[tag]
        if verbose:
            print(f'    -> Widened to {len(filtered)} candidates')

    # -- Stage 2: Matcher (o3-mini) --
    if verbose:
        print('    [3/4] Matching best directions (o3-mini)...', end=' ')
    match_result = match_directions(question_text, reasoner_result, filtered)
    if match_result is None:
        print('FAILED')
        return None
    if verbose:
        dirs = [d['direction_id'] for d in match_result['selected_directions']]
        print(f'OK -> {dirs}')

    # -- Stage 3: Generator (gpt-4o) --
    if verbose:
        print('    [4/4] Generating essay (gpt-4o)...', end=' ')

    # Build filtered Section 1 with ONLY the selected directions
    sec1_parts = []
    for d in match_result['selected_directions']:
        did = d['direction_id']
        info = DIRECTION_CATALOG.get(did, {})
        sec1_parts.append(
            f"DIRECTION: {did}\n"
            f"  Assigned to: {d['assigned_to']}\n"
            f"  ARGUMENT: {info.get('argument', 'N/A')}\n"
            f"  LOGIC: {info.get('logic', 'N/A')}"
        )
    for fp in match_result.get('freestyle_paragraphs', []):
        sec1_parts.append(f"FREESTYLE: {fp} -- write using your own reasoned argument, direction_tag: null")

    filtered_section1 = '\n\n'.join(sec1_parts)

    system_prompt = (
        REQUIRED_DOCS['Writer.md']
        + "\n\n=== PRE-DECIDED (do not change) ==="
        + f"\nStructure: {match_result['structure_type']}"
        + f"\nEssay plan: {match_result['essay_plan']}"
        + "\n\n=== YOUR ASSIGNED DIRECTIONS ==="
        + f"\n{filtered_section1}"
        + "\n\n=== RHETORIC TAGS ==="
        + f"\n{REF_SEC2_RHETORIC}"
        + "\n\n=== STRUCTURE TEMPLATES ==="
        + f"\n{REF_SEC3_TEMPLATES}"
    )

    user_message = (
        f"Write a complete IELTS essay for this question, then decompose and tag it:\n\n"
        f"{question_text}\n\n"
        "Follow Steps 3 through 7 in your instructions: Write (Step 3) -> Traceability Check (Step 4) -> Decompose (Step 5) -> Tag (Step 6) -> Extract Lexical Items (Step 7).\n"
        "Seed your prose from each direction's LOGIC field.\n"
        "Output only the final JSON."
    )

    for attempt in range(max_retries + 1):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_message}
                ],
                max_tokens=16000,
                temperature=0.4,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            if not content or not content.strip():
                raise ValueError("Model returned empty response")
            data = json.loads(content)
            essay = EssayOutput(**data)
            if verbose:
                print(f'OK [{len(essay.sentences)} sentences]')
            return essay
        except Exception as e:
            if attempt < max_retries:
                print(f'    ⚠️ Generator attempt {attempt+1}: {e}. Retrying...')
                time.sleep(2)
            else:
                if verbose:
                    print(f'FAILED: {e}')
                return None


print('✅ 4-stage generate_essay() ready')
print('   Reasoner:  gpt-4o  (question -> activated poles)')
print('   Lookup:    Python  (poles -> ~20 candidate directions)')
print('   Matcher:   o3-mini (candidates + LOGIC -> 2-3 best picks)')
print('   Generator: gpt-4o  (focused essay writing + tagging)')

## 👁️ Station 2.5 — Spot Check (First 5 Essays)

In [ ]:
def display_essay_for_review(q_number: int, q_text: str, essay: EssayOutput):
    """Pretty-print an essay for human review."""
    print(f"\n{'='*70}")
    print(f"Q{q_number}: {q_text}")
    print(f"Structure: {essay.structure_type}")
    print(f"{'='*70}")

    current_para = None
    for s in essay.sentences:
        if s.paragraph_type != current_para:
            current_para = s.paragraph_type
            print(f"\n  [{current_para.upper()}]")

        direction_display = s.direction_tag or 'null'
        print(f"  [{s.rhetoric_tag}] | dir: {direction_display}")
        print(f"  {s.text}")
        lexicals = ', '.join([li.phrase for li in s.lexical_items])
        print(f"  💡 Lexical: {lexicals}")
        print()

# Run spot check on first 5 questions
print('🔍 SPOT CHECK — Generating first 5 essays for your review...')
print('This may take 1-2 minutes.\n')

spot_check_results = []  # store (question, essay) tuples

for q in questions[:5]:
    print(f"Generating Q{q['number']}...")
    essay = generate_essay(q['text'])
    if essay:
        spot_check_results.append((q, essay))
        display_essay_for_review(q['number'], q['text'], essay)
    else:
        print(f"  ❌ Q{q['number']} failed to generate")
    time.sleep(1)  # gentle rate limiting

print(f"\n{'='*70}")
print(f'✅ Spot check complete. {len(spot_check_results)}/5 essays generated.')
print('\nReview the essays above, then run the next cell to decide.')

🔍 SPOT CHECK — Generating first 5 essays for your review...
This may take 1-2 minutes.

Generating Q1...

Q1: Some people think society could benefit more if more people study business than history. To what extent do you agree or disagree?
Structure: structure_3

  [INTRODUCTION]
  [paraphrase_question] | dir: null
  There is an ongoing debate about whether studying business offers more societal benefits than studying history.
  💡 Lexical: ongoing debate, societal benefits

  [clear_thesis_statement] | dir: null
  I disagree with this notion because history provides crucial insights into cultural identity and prevents the repetition of past mistakes.
  💡 Lexical: crucial insights, cultural identity


  [BODY_1]
  [topic_sentence] | dir: individual_preservation_reverse
  History plays a vital role in shaping cultural heritage and identity.
  💡 Lexical: vital role, cultural heritage

  [explanation] | dir: individual_preservation_reverse
  When societies understand their past, they devel

In [ ]:
# ✋ HUMAN DECISION POINT
# Review the essays printed above.
# Type 'proceed' to continue with all 100 questions.
# Type 'abort' to stop and investigate.

decision = input("\nType 'proceed' to run full batch, or 'abort' to stop: ").strip().lower()

if decision == 'proceed':
    print('✅ Proceeding to full batch.')
    PROCEED = True
else:
    print('🛑 Aborted. Fix any issues then re-run the spot check cell.')
    PROCEED = False


Type 'proceed' to run full batch, or 'abort' to stop: proceed
✅ Proceeding to full batch.


## ⚙️ Station 3 & 4 — Embed + Write to Supabase

In [ ]:
import uuid

def write_essay_to_supabase(question: dict, essay: EssayOutput, batch_id: str) -> bool:
    """
    Embeds and writes one complete essay to Supabase.
    Write order: sentences → lexical_items → syntax_items → sentence_syntax → essays → essay_sentences
    Returns True on success, False on any error.
    """
    try:
        sentence_ids = []
        syntax_cache = {}  # pattern_text -> syntax_id (avoid duplicate inserts in same essay)

        for s in essay.sentences:
            # ── Embed the sentence ──────────────────────────────────────────
            sentence_embedding = embed(s.text)
            word_count = len(s.text.split())

            # ── Write sentence row ──────────────────────────────────────────
            sentence_row = {
                'sentence_id':    str(uuid.uuid4()),
                'canonical_text': s.text,
                'embedding':      sentence_embedding,
                'rhetoric_tag':   s.rhetoric_tag,
                'direction_tag':  s.direction_tag,
                'word_count':     word_count,
                'batch_id':       batch_id,
            }
            res = supabase.table('sentences').insert(sentence_row).execute()
            sentence_id = res.data[0]['sentence_id']
            sentence_ids.append({'id': sentence_id, 'paragraph_type': s.paragraph_type, 'order': s.order})

            # ── Write lexical_items ─────────────────────────────────────────
            for li in s.lexical_items:
                lex_embedding = embed(li.phrase)
                supabase.table('lexical_items').insert({
                    'lexical_id':  str(uuid.uuid4()),
                    'sentence_id': sentence_id,
                    'phrase':      li.phrase,
                    'embedding':   lex_embedding,
                }).execute()

            # ── Write syntax_items & junction ──────────────────────────────
            if s.syntax_pattern:
                pattern = s.syntax_pattern
                if pattern not in syntax_cache:
                    syntax_embedding = embed(pattern)
                    res2 = supabase.table('syntax_items').insert({
                        'syntax_id': str(uuid.uuid4()),
                        'pattern':   pattern,
                        'embedding': syntax_embedding,
                    }).execute()
                    syntax_cache[pattern] = res2.data[0]['syntax_id']

                supabase.table('sentence_syntax').insert({
                    'id':          str(uuid.uuid4()),
                    'sentence_id': sentence_id,
                    'syntax_id':   syntax_cache[pattern],
                }).execute()

        # ── Write essay row ─────────────────────────────────────────────────
        total_words = sum(len(s.text.split()) for s in essay.sentences)
        res3 = supabase.table('essays').insert({
            'essay_id':       str(uuid.uuid4()),
            'question':       question['text'],
            'structure_type': essay.structure_type,
            'word_count':     total_words,
            'batch_id':       batch_id,
        }).execute()
        essay_id = res3.data[0]['essay_id']

        # ── Write essay_sentences junction rows ─────────────────────────────
        for s_data in sentence_ids:
            supabase.table('essay_sentences').insert({
                'id':             str(uuid.uuid4()),
                'essay_id':       essay_id,
                'sentence_id':    s_data['id'],
                'paragraph_type': s_data['paragraph_type'],
                'order':          s_data['order'],
            }).execute()

        return True

    except Exception as e:
        print(f'    ❌ DB write error: {e}')
        return False

print('✅ DB writer function ready')

✅ DB writer function ready


## 🚀 Station — Full Batch Run (Questions 1–100)

In [ ]:
if not PROCEED:
    print('🛑 Batch aborted. Run the spot check and proceed cell first.')
else:
    import datetime

    batch_id = str(uuid.uuid4())
    print(f'🚀 Starting full batch run')
    print(f'   Batch ID : {batch_id}')
    print(f'   Questions: {len(questions)}')
    print(f'   Model    : gpt-4o')
    print(f'   Started  : {datetime.datetime.now().strftime("%H:%M:%S")}')
    print()

    # The first 5 are already generated in spot check — reuse them
    successes = []
    failures  = []

    # Write spot check results first (already generated)
    print('Writing spot check essays (Q1–Q5) to Supabase...')
    for q, essay in spot_check_results:
        ok = write_essay_to_supabase(q, essay, batch_id)
        if ok:
            successes.append(q['number'])
            print(f'  ✅ Q{q["number"]} written')
        else:
            failures.append(q['number'])
            print(f'  ❌ Q{q["number"]} write failed')

    # Generate and write remaining questions (Q6–Q100)
    print(f'\nGenerating and writing Q6–Q{len(questions)}...')
    for q in questions[5:]:
        print(f'  Processing Q{q["number"]}...', end=' ')

        essay = generate_essay(q['text'])
        if essay is None:
            failures.append(q['number'])
            print('❌ generation failed')
            continue

        ok = write_essay_to_supabase(q, essay, batch_id)
        if ok:
            successes.append(q['number'])
            print('✅')
        else:
            failures.append(q['number'])
            print('❌ write failed')

        time.sleep(0.5)  # gentle rate limiting

    # ── Final Report ────────────────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'BATCH COMPLETE — {datetime.datetime.now().strftime("%H:%M:%S")}')
    print(f'  Batch ID  : {batch_id}')
    print(f'  Successes : {len(successes)}/{len(questions)}')
    print(f'  Failures  : {len(failures)}')
    if failures:
        print(f'  Failed Qs : {sorted(failures)}')
    print(f'{"="*60}')

🚀 Starting full batch run
   Batch ID : 6464ca79-871b-4c91-a5ee-6a6f24dbf5eb
   Questions: 100
   Model    : gpt-4o
   Started  : 12:33:32

Writing spot check essays (Q1–Q5) to Supabase...
  ✅ Q1 written
  ✅ Q2 written
  ✅ Q3 written
  ✅ Q4 written
  ✅ Q5 written

Generating and writing Q6–Q100...
  Processing Q6...     ⚠️  Attempt 1 failed: 8 validation errors for EssayOutput
sentences.2.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
sentences.3.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
sentences.4.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value

## 🔁 Optional — Retry Failed Questions

In [ ]:
# Run this cell only if there were failures above
# It will retry only the failed question numbers

if 'failures' in dir() and failures:
    print(f'Retrying {len(failures)} failed questions: {sorted(failures)}')
    retry_successes = []
    still_failed    = []

    failed_questions = [q for q in questions if q['number'] in failures]

    for q in failed_questions:
        print(f'  Retrying Q{q["number"]}...', end=' ')
        essay = generate_essay(q['text'], max_retries=3)
        if essay is None:
            still_failed.append(q['number'])
            print('❌')
            continue
        ok = write_essay_to_supabase(q, essay, batch_id)
        if ok:
            retry_successes.append(q['number'])
            print('✅')
        else:
            still_failed.append(q['number'])
            print('❌')
        time.sleep(1)

    print(f'\nRetry result: {len(retry_successes)} recovered, {len(still_failed)} still failing')
    if still_failed:
        print(f'Still failed: {sorted(still_failed)}')
else:
    print('No failures to retry. All good!')

Retrying 55 failed questions: [24, 27, 28, 29, 31, 36, 37, 38, 43, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91]
  Retrying Q24... ✅
  Retrying Q27... ✅
  Retrying Q28...     ⚠️  Attempt 1 failed: 11 validation errors for EssayOutput
sentences.2.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
sentences.3.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
sentences.4.paragraph_type
  Value error, Invalid paragraph_type: 'body' [type=value_error, input_value='body', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
s

## 🔎 Station — Verify DB (Quick Check)

In [ ]:
# Quick count of rows in each table to confirm writes worked
tables = ['essays', 'essay_sentences', 'sentences', 'lexical_items', 'syntax_items', 'sentence_syntax']

print('📊 Row counts in Supabase:')
for table in tables:
    try:
        res   = supabase.table(table).select('*', count='exact').execute()
        count = res.count
        print(f'  {table:<20} : {count} rows')
    except Exception as e:
        print(f'  {table:<20} : ERROR — {e}')

📊 Row counts in Supabase:
  essays               : 100 rows
  essay_sentences      : 1284 rows
  sentences            : 1284 rows
  lexical_items        : 2593 rows
  syntax_items         : 1284 rows
  sentence_syntax      : 1284 rows
